<a href="https://colab.research.google.com/github/ewilpeers/bible/blob/master/xG_Collab/01GKSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# mēķis: atrast visas vietas kur šis minēts (takā konkordance līdzīgi)

# 1 -> DEFinīcijas

In [1]:
class GithubSecretsTikParaizsTikDrosh():
    def encode(sl, s: str) -> bytes:
        # Rotate each UTF-8 byte right by 1. Returns bytes (not str) so the
        # payload can't be mangled by text-encoding round-trips.
        result = bytearray()
        for b in s.encode('utf-8'):
            result.append(((b & 1) << 7) | (b >> 1))
        return bytes(result)

    def encode_escaped(sl, s: str) -> str:
        # Returns a Python bytes literal you can paste into source,
        # e.g. b'\x8a\xbc...'
        encoded = sl.encode(s)
        return "b'" + ''.join(f'\\x{b:02x}' for b in encoded) + "'"

    def decode(sl, s) -> str:
        # Accepts bytes (preferred) or a latin-1 str for backwards-compat.
        if isinstance(s, str):
            s = s.encode('latin-1')
        result = bytearray()
        for b in s:
            result.append(((b & 0x80) >> 7) | ((b << 1) & 0xFF))
        return result.decode('utf-8')

    def print_assignment(sl, name: str, s: str) -> None:
        # Prints a ready-to-paste line: NAME = b'\x..\x..'
        print(f"{name} = {sl.encode_escaped(s)}")


gh = GithubSecretsTikParaizsTikDrosh()

In [2]:
import pandas as pd
TOKEN_NT_DATA_FU_GITHUB = b'\xb3\xb4\x3a\x34\xba\x31\xaf\x38\xb0\x3a\xaf\x98\x98\x21\x29\x9a\x23\xa2\x23\xa4\x18\xb2\x38\x19\x39\xa0\xbc\x1c\x3a\xac\x24\xb2\x35\xaf\xa8\x9c\x18\x9c\xb3\xa1\xb8\xa2\xa2\xb2\x26\xb9\x19\x2d\x31\x3c\x39\x35\xb3\x21\xac\x26\xa1\x3d\x29\xa9\xb2\x2a\xb8\x34\x1c\x29\x32\xb8\xa1\xa4\xa1\x18\xac\xa8\xa8\x3b\x37\x26\xa6\xaa\xa4\x25\x23\xa6\x24\x9b\xa2\x33\x34\xa5\xb1\x2c\x1b'
TOKEN_NT_DATA = gh.decode(TOKEN_NT_DATA_FU_GITHUB)
TOKEN_OT_DATA_FU_GITHUB = b'\xb3\xb4\x3a\x34\xba\x31\xaf\x38\xb0\x3a\xaf\x98\x98\xa0\xaa\x28\xa2\x9a\x9b\xa8\x18\xbc\x32\x9a\xa7\xb0\xb3\xb3\x33\x19\xa9\xb0\x3d\xaf\xa3\x3c\xb4\xa5\x9b\x39\x1a\x9a\x36\x3a\x99\x9a\x2b\xb4\x2c\xb6\x19\xb6\xa7\x3a\xac\xa0\xa5\xa3\x38\x33\x3b\x18\xb9\x38\xb0\x2c\x39\xbb\xb2\xb9\x99\xa1\xb3\xaa\xa7\xa6\xb4\x9a\x2d\x2c\x23\x24\xa8\xa9\xa5\xb4\x3d\x18\xb2\x25\x3d\x32\x39'
TOKEN_OT_DATA = gh.decode(TOKEN_OT_DATA_FU_GITHUB)
NT_REPO_DATA_PATH = "ewilpeers/new-testament-data"
OT_REPO_DATA_PATH = "grauziitisos/ot-west-len-data"

In [3]:

import requests
import pandas as pd
def download_csv_from_private_repo(repopath, path_in_repo, token, *args, **kwargs):
	url = f"https://api.github.com/repos/{repopath}/contents/{path_in_repo}"
	r = requests.get(url, headers={
		"Authorization": f"token {token}",
		"Accept": "application/vnd.github.v3.raw"
	})
	r.raise_for_status()
	from io import StringIO
	import pandas as pd
	return pd.read_csv(StringIO(r.text), *args, **kwargs)

In [4]:
import requests
import pandas as pd
from io import BytesIO

def download_csv_from_release(repopath, tag, filename, token=None, *args, **kwargs):
    if token:
        # Private repo: go through the API with auth.
        # First, find the asset ID for this filename within the release.
        api_url = f"https://api.github.com/repos/{repopath}/releases/tags/{tag}"
        meta = requests.get(api_url, headers={
            "Authorization": f"token {token}",
            "Accept": "application/vnd.github+json",
        })
        meta.raise_for_status()
        assets = meta.json()["assets"]
        asset = next((a for a in assets if a["name"] == filename), None)
        if asset is None:
            raise FileNotFoundError(
                f"{filename!r} not found in release {tag!r}. "
                f"Available: {[a['name'] for a in assets]}"
            )

        # Now fetch the asset bytes. octet-stream is the magic header.
        blob = requests.get(asset["url"], headers={
            "Authorization": f"token {token}",
            "Accept": "application/octet-stream",
        })
        blob.raise_for_status()
        return pd.read_csv(BytesIO(blob.content), *args, **kwargs)


    url = f"https://github.com/{repopath}/releases/download/{tag}/{filename}"
    return pd.read_csv(url, *args, **kwargs)


In [5]:
import os
def _cached_repo(repo_path, token, path_in_repo, filename=None, **kwargs):
    """Download CSV from a private repo, or load from local cache."""
    filename = filename or path_in_repo.split('/')[-1]
    if os.path.exists(filename):
        return pd.read_csv(filename, dtype=kwargs.get('dtype'))
    df = download_csv_from_private_repo(repo_path, path_in_repo, token, **kwargs)
    df.to_csv(filename, index=False)
    return df

def _cached_release(repo_path, token, release_tag, path_in_repo, filename=None, **kwargs):
    """Download CSV from a release, or load from local cache."""
    filename = filename or path_in_repo.split('/')[-1]
    if os.path.exists(filename):
        return pd.read_csv(filename, dtype=kwargs.get('dtype'))
    df = download_csv_from_release(repo_path, release_tag, path_in_repo, token, **kwargs)
    df.to_csv(filename, index=False)
    return df

In [6]:
l65_df         = _cached_repo(OT_REPO_DATA_PATH, TOKEN_OT_DATA, "RT65_words.csv",                     dtype={'strong_num': str})
l24_df         = _cached_repo(OT_REPO_DATA_PATH, TOKEN_OT_DATA, "JTR2024_words.csv",                  dtype={'strong_num': str})
hb_df          = _cached_release(OT_REPO_DATA_PATH, TOKEN_OT_DATA, "data-v1", "biblehub_hb_en_direct.csv", dtype={'strong_num': str})
strongs_df     = _cached_repo(OT_REPO_DATA_PATH, TOKEN_OT_DATA, "greek-enh-lxx-apo-OT-blb.csv",       dtype={'strong_num': str})
abp_strongs_df = _cached_repo(OT_REPO_DATA_PATH, TOKEN_OT_DATA, "biblehub_LXX_EXT_el_en_direct.csv",  dtype={'strong_num': str})
l1694_df       = _cached_repo(OT_REPO_DATA_PATH, TOKEN_OT_DATA, "GL1694_words.csv")

/tmp/ipykernel_6068/547696060.py:12: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(StringIO(r.text), *args, **kwargs)


In [7]:
nt_strongs_df = _cached_repo(NT_REPO_DATA_PATH, TOKEN_NT_DATA, "strongs.csv",         filename="nt_strongs.csv")
nt_lv65_df    = _cached_repo(NT_REPO_DATA_PATH, TOKEN_NT_DATA, "latvian_full65.csv")
nt_l24_df     = _cached_repo(NT_REPO_DATA_PATH, TOKEN_NT_DATA, "JTR2024_words.csv",   filename="nt_JTR2024_words.csv", dtype={'strong_num': str})
nt_l1694_df   = _cached_repo(NT_REPO_DATA_PATH, TOKEN_NT_DATA, "GL1694_words.csv",    filename="nt_GL1694_words.csv")

In [8]:
hb_df.head()

,verse,word,form,form_en,strong_num,strong_title,strong_en_title,translit,translit_title,book,chapter
0,1,0,וַיֹּ֨אמֶר,And said,559,"559: 1) to say, speak, utter <BR> 1a) (Qal) to...",Conj‑w|V‑Qal‑ConsecImperf‑3ms,way·yō·mer,vai·Yo·mer: And said -- Occurrence 1898 of 1948.,hosea,3
1,1,1,יְהוָ֜ה,Yahweh,3068,3068: Jehovah = the existing One<BR> 1) the pr...,N‑proper‑ms,Yah·weh,Yah·weh: Yahweh -- Occurrence 5792 of 6218.,hosea,3
2,1,2,אֵלַ֗י,to me,413,"413: 1) to, toward, unto (of motion) <BR> 2) i...",Prep|1cs,"’ê·lay,","'e·Lai,: to me -- Occurrence 415 of 446.",hosea,3
3,1,3,ע֚וֹד,again,5750,"5750: subst<BR> 1) a going round, continuance ...",Adv,‘ō·wḏ,od: again -- Occurrence 363 of 405.,hosea,3
4,1,4,לֵ֣ךְ,go,1980,"1980: 1) to go, walk, come <BR> 1a) (Qal) <BR>...",V‑Qal‑Imp‑ms,lêḵ,lech: go -- Occurrence 87 of 91.,hosea,3


In [9]:
nt_strongs_g = nt_strongs_df.groupby(["book", "chapter", "verse"])
nt_lv_g = nt_lv65_df.groupby(["book", "chapter", "verse"])
nt_l24_g = nt_l24_df.groupby(["book", "chapter", "verse"])
nt_l1694_g = nt_l1694_df.groupby(["book", "chapter", "verse"])

l65_g = l65_df.groupby(["book", "chapter", "verse"])
l24_g = l24_df.groupby(["book", "chapter", "verse"])
hb_g = hb_df.groupby(["book", "chapter", "verse"])
strongs_g = strongs_df.groupby(["book", "chapter", "verse"])
abp_strongs_g = abp_strongs_df.groupby(["book", "chapter", "verse"])
l1694_g = l1694_df.groupby(["book", "chapter", "verse"])

In [10]:
import urllib.request, zipfile, io, os, pathlib

OT_URL = "https://github.com/ewilpeers/bible/releases/download/data-v0/bible_ot.zip"
NT_URL = "https://github.com/ewilpeers/new-testament/releases/download/data-v0/bible_nt.zip"
OT_DIR = 'bible/ot'
NT_DIR = 'bible/nt'
def fetch_and_extract(url, dest):
    dest = pathlib.Path(dest)
    if dest.exists() and any(dest.iterdir()):
        return dest  # already cached
    dest.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(url) as r:
        zipfile.ZipFile(io.BytesIO(r.read())).extractall(dest)
    return dest

ot_dir = None
if not os.path.exists(OT_DIR):
  ot_dir = fetch_and_extract(OT_URL, OT_DIR)
else:
  ot_dir =  pathlib.Path(OT_DIR)
nt_dir = None
if not os.path.exists(NT_DIR):
  nt_dir = fetch_and_extract(NT_URL, NT_DIR)
else:
  nt_dir = pathlib.Path(NT_DIR)

In [11]:
import json
records = []
leftovers = []
for book_dir in sorted(nt_dir.iterdir()):
    if not book_dir.is_dir():
        continue
    book = book_dir.name
    for json_file in sorted(
        [f for f in book_dir.glob('*.json') if f.stem.isdigit()],
        key=lambda x: int(x.stem)
    ):
        chapter = int(json_file.stem)
        try:
            verses = json.loads(json_file.read_text(encoding='utf-8'))
        except Exception as e:
            print(f"  skip {json_file}: {e}")
            continue

        for i, verse in enumerate(verses):
            for j, gw in enumerate(verse.get('greek_words', [])):
                records.append({
                    'book': book,
                    'chapter': chapter,
                    'verse': i+1,
                    'word': j, #vardi ir 0-bazeti neka panti
                    'greek': gw.get('greek', []),
                    'latvian': gw.get('latvian', []),
                })
            leftovers.append(
                {
                    'book': book,
                    'chapter': chapter,
                    'verse': i+1,
                    'leftover_latvian': verse.get('leftover_latvian', [])
                }
            )

nt_df = pd.DataFrame(records)
nt_leftovers_df = pd.DataFrame(leftovers)

In [12]:
records = []
leftovers_lv = []
leftovers_gk = []
for book_dir in sorted(ot_dir.iterdir()):
    if not book_dir.is_dir():# or not book_dir.name=='1_samuel':
        continue
    book = book_dir.name
    for json_file in sorted(
        [f for f in book_dir.glob('*.json') if f.stem.isdigit()], #and f.stem=='20'],
        key=lambda x: int(x.stem)
    ):
        chapter = int(json_file.stem)
        try:
            verses = json.loads(json_file.read_text(encoding='utf-8'))
        except Exception as e:
            print(f"  skip {json_file}: {e}")
            continue

        for i, verse in enumerate(verses):
            for j, hw in enumerate(verse.get('hebrew_words', [])):
              #shis uzkars uz paris minutem visu lapu, bet izvadiis arii visu VD :D
                #print(hw)
                records.append({
                  'book': book,
                  'chapter': chapter,
                  'verse': i+1,
                  'word': j,
                  'hebrew': hw.get('hebrew', []),
                  'greek': hw.get('greek', []),
                  'latvian': hw.get('latvian', [])
                })

            leftovers_lv.append({
                    'book': book,
                    'chapter': chapter,
                    'verse': i+1,
                    'leftover_latvian': verse.get('leftover_latvian', [])
                })
            leftovers_gk.append({
                    'book': book,
                    'chapter': chapter,
                    'verse': i+1,
                    'leftover_latvian': verse.get('leftover_greek', [])
                })




ot_df = pd.DataFrame(records)
ot_leftovers_lv_df = pd.DataFrame(leftovers_lv)
ot_leftovers_gk_df = pd.DataFrame(leftovers_gk)

## utils

In [13]:
_LV_TABLE = str.maketrans('āčēģīķļņõŗšūžĀČĒĢĪĶĻŅÕŖŠŪŽ',
                          'acegiklnorsuzACEGIKLNORSUZ')

def strip_latvian_deep(s: str) -> str:
    return s.lower().translate(_LV_TABLE)

In [14]:
class searchExecutorSimple():
  def __init__(s, df, transFunc, case=False, regex=False):
    s.df = df
    s.case = case
    s.regex = regex
    s.transFunc = transFunc
  def search(s, searchStr):
    if s.case:
      return s.df[s.df['form'].apply(s.transFunc).str.contains(s.transFunc(searchStr), case=False)]
    else:
      return s.df[s.df['form'].apply(s.transFunc).str.contains(s.transFunc(searchStr), case=False)]

In [15]:
def verse_set(df, mapper = lambda x: x):
    return set(zip(df['book'], df['chapter'], df['verse']))

## MAPpings

In [16]:
#split strings ir pirmaaas daljas nobeigums, ieskaitot, naakamaa dalja ir peec taa ko noraada
# l65_to_hb_merge[(('1_chronicles', 12, 4), "trīsdesmit")] = (('1_chronicles', 12, 4), ('1_chronicles', 12, 5))
# pirmaa dala ietvers sevii visu liidz vaardam triisdesmit ieskaitot (12,4), otraa tuuliit aiz taa triisdesmit(12,5)
# lv vietturis : ebreju vietturis
l65_to_hb_merge = {}
l65_to_hb={
('1_chronicles', 5, 27) : ('1_chronicles', 6, 1),
('1_chronicles', 5, 28) : ('1_chronicles', 6, 2),
('1_chronicles', 5, 29) : ('1_chronicles', 6, 3),
('1_chronicles', 5, 30) : ('1_chronicles', 6, 4),
('1_chronicles', 5, 31) : ('1_chronicles', 6, 5),
('1_chronicles', 5, 32) : ('1_chronicles', 6, 6),
('1_chronicles', 5, 33) : ('1_chronicles', 6, 7),
('1_chronicles', 5, 34) : ('1_chronicles', 6, 8),
('1_chronicles', 5, 35) : ('1_chronicles', 6, 9),
('1_chronicles', 5, 36) : ('1_chronicles', 6, 10),
('1_chronicles', 5, 37) : ('1_chronicles', 6, 11),
('1_chronicles', 5, 38) : ('1_chronicles', 6, 12),
('1_chronicles', 5, 39) : ('1_chronicles', 6, 13),
('1_chronicles', 5, 40) : ('1_chronicles', 6, 14),
('1_chronicles', 5, 41) : ('1_chronicles', 6, 15)
}
for i in range(1, 66+1):
    l65_to_hb[('1_chronicles', 6, i)]=('1_chronicles', 6, i+15)

l65_to_hb_merge[(('1_chronicles', 12, 4), "tiem trīsdesmit")] = (('1_chronicles', 12, 4), ('1_chronicles', 12, 5))
for i in range(5, 39+1):
    l65_to_hb[('1_chronicles', 12, i)] = ('1_chronicles', 12, i+1)
l65_to_hb_merge [(('1_chronicles', 12, 39), ('1_chronicles', 12, 40))] = ('1_chronicles', 12, 40)


for l, h in zip(range(1, 14+1), range(21, 34+1)):
    l65_to_hb[('1_kings', 5, l)]=('1_kings', 4, h)

for i in range(15, 32+1):
    l65_to_hb[('1_kings', 5, i)]=('1_kings', 5, i-14)



#merge 43 + 44
l65_to_hb_merge[(('1_kings', 22, 43), ('1_kings', 22, 44))]= ('1_kings', 22, 43)

#now update mapping
for i in range(43, 53+1):
    l65_to_hb[('1_kings', 22, i+1)]=('1_kings', 22, i)
#1 samuel 20:42+
l65_to_hb[('1_samuel', 20, 43)]=('1_samuel', 21, 1)
for i in range(1, 13+1):
    l65_to_hb[('1_samuel', 21, i)]=('1_samuel', 21, i+1)
#14, 15 merged => 14 hebrw
l65_to_hb_merge[(('1_samuel', 21, 14), ('1_samuel', 21, 15))] = ('1_samuel', 21, 15)

#1 samuel 23:28+
l65_to_hb[('1_samuel', 24, 1)] = ('1_samuel', 23, 29)
for i in range(2, 23+1):
    l65_to_hb[('1_samuel', 24, i)]=('1_samuel', 24, i-1)

#2chronicles 2:1
l65_to_hb[('2_chronicles', 1, 18)] = ('2_chronicles', 2, 1)
for i in range(1, 17+1):
    l65_to_hb[('2_chronicles', 2, i)]=('2_chronicles', 2, i+1)

#2chronicles 14:1
l65_to_hb[('2_chronicles', 13, 23)] = ('2_chronicles', 14, 1)
for i in range(1, 14+1):
    l65_to_hb[('2_chronicles', 14, i)]=('2_chronicles', 14, i+1)

#2kings 12:1-21
l65_to_hb[('2_kings', 12, 1)] = ('2_kings', 11, 21)
for i in range(2, 22+1):
    l65_to_hb[('2_kings', 12, i)]=('2_kings', 12, i-1)

#2 sam 18:33
l65_to_hb[('2_samuel', 19, 1)] = ('2_samuel', 18, 33)
for i in range(2, 44+1):
    l65_to_hb[('2_samuel', 19, i)]=('2_samuel', 19, i-1)

#daniel 3
l65_to_hb[('daniel', 3, 31)] = ('daniel', 4, 1)
#obsolote, now merge ir rghtfully updating the map.l65_to_hb[('daniel', 3, 32)] = ('daniel', 4, 2), 3

#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!1 O T R Ā D I !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# 1 lv verse = 2x heb verses, TODO: SPLIT THEM ON WORD BEING LAST IN 1st part
l65_to_hb_merge[(('daniel', 3, 32), "darījis")] = (('daniel', 4, 2), ('daniel', 4, 3))
for i in range(1, 34+1):
    l65_to_hb[('daniel', 4, i)] = ('daniel', 4, i+3)

#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!1 O T R Ā D I !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# 1 lv verse = 2x heb verses, TODO: SPLIT THEM ON WORD BEING LAST IN 1st part
l65_to_hb_merge[(('daniel', 5, 30), "nogalināts")] = (('daniel', 5, 30), ('daniel', 5, 31))

l65_to_hb[('deuteronomy', 13, 1)] = ('deuteronomy', 12, 32)
for i in range(2, 19+1):
    l65_to_hb[('deuteronomy', 13, i)]=('deuteronomy', 13, i-1)

l65_to_hb[('deuteronomy', 23, 1)] = ('deuteronomy', 22, 30)
for i in range(2, 26+1):
    l65_to_hb[('deuteronomy', 23, i)]=('deuteronomy', 23, i-1)

l65_to_hb[('deuteronomy', 28, 69)] = ('deuteronomy', 29, 1)
l65_to_hb_merge[(('deuteronomy', 29, 1), ('deuteronomy', 29, 2))]=('deuteronomy', 29, 2)

#ecclesiates
l65_to_hb[('ecclesiastes', 4, 17)] = ('ecclesiastes', 5, 1)
for i in range(1, 19+1):
    l65_to_hb[('ecclesiastes', 5, i)]=('ecclesiastes', 5, i+1)

l65_to_hb[('exodus', 1, 21)] = ('exodus', 1, 22)

l65_to_hb[('exodus', 7, 26)] = ('exodus', 8, 1)
l65_to_hb[('exodus', 7, 27)] = ('exodus', 8, 2)
l65_to_hb[('exodus', 7, 28)] = ('exodus', 8, 3)
l65_to_hb[('exodus', 7, 29)] = ('exodus', 8, 4)
for i in range(1, 28+1):
    l65_to_hb[('exodus', 8, i)]=('exodus', 8, i+4)

l65_to_hb[('exodus', 21, 37)] = ('exodus', 22, 1)
for i in range(1, 30+1):
    l65_to_hb[('exodus', 22, i)]=('exodus', 22, i+1)
#ezekiel
for i in range(45, 49+1):
    l65_to_hb[('ezekiel', 21, i-44)]=('ezekiel', 20, i)
for i in range(6, 37+1):
    l65_to_hb[('ezekiel', 21, i)]=('ezekiel', 21, i-5)

# genesis 3 panti savienoti vienā un izlaista 1 tauta
l65_to_hb_merge[(('genesis', 15, 19), "kadmoniešus", "refajiešus")] = (('genesis', 15, 19), ('genesis', 15, 20), ('genesis', 15, 21))

l65_to_hb_merge[(('genesis', 24, 65),('genesis', 24, 66))] = ('genesis', 24, 65)
for i in range(67, 68+1):
    l65_to_hb[('genesis', 24, i)]=('genesis', 24, i-1)

l65_to_hb[('genesis', 32, 1)]=('genesis', 31, 55)
for i in range(2, 33+1):
    l65_to_hb[('genesis', 32, i)]=('genesis', 32, i-1)

#hosea 1
l65_to_hb[('hosea', 2, 1)] = ('hosea', 1, 10)
l65_to_hb[('hosea', 2, 2)] = ('hosea', 1, 11)
for i in range(3, 25+1):
    l65_to_hb[('hosea', 2, i)]=('hosea', 2, i-2)

l65_to_hb[('hosea', 12, 1)] = ('hosea', 11, 12)
for i in range(2, 15+1):
    l65_to_hb[('hosea', 12, i)]=('hosea', 12, i-1)

l65_to_hb[('hosea', 14, 1)] =('hosea', 13, 16)
for i in range(2, 10+1):
    l65_to_hb[('hosea', 14, i)]=('hosea', 14, i-1)

#isaiah
l65_to_hb[('isaiah', 8, 23)] = ('isaiah', 9, 1)
for i in range(1, 20+1):
    l65_to_hb[('isaiah', 9, i)]=('isaiah', 9, i+1)

l65_to_hb[('jeremiah', 8, 23)] = ('jeremiah', 9, 1)
for i in range(1, 25+1):
    l65_to_hb[('jeremiah', 9, i)]=('jeremiah', 9, i+1)

# job
for i in range(25, 32+1):
    l65_to_hb[('job', 40, i)]=('job', 41, i-24)
for i in range(1, 26+1):
    l65_to_hb[('job', 41, i)]=('job', 41, i+8)
#joel
for i in range(1, 5+1):
    l65_to_hb[('joel', 3, i)]=('joel', 2, i+27)
for i in range(1, 21+1):
    l65_to_hb[('joel', 4, i)]=('joel', 3, i)

#jonah
l65_to_hb[('jonah', 2, 1)] = ('jonah', 1, 17)
for i in range(2, 11+1):
    l65_to_hb[('jonah', 2, i)]=('jonah', 2, i-1)

#leviticus
for i in range(20, 26+1):
    l65_to_hb[('leviticus', 5, i)]=('leviticus', 6, i-19)
for i in range(1, 23+1):
    l65_to_hb[('leviticus', 6, i)]=('leviticus', 6, i+7)

#malachi
for i in range(19, 24+1):
    l65_to_hb[('malachi', 3, i)]=('malachi', 4, i-18)
#malachi 4 in Latvian does not exist!!

#micah
l65_to_hb[('micah', 4, 14)] = ('micah', 5, 1)
for i in range(1, 14+1):
    l65_to_hb[('micah', 5, i)]=('micah', 5, i+1)

#nahum
l65_to_hb[('nahum', 2, 1)] = ('nahum', 1, 15)
for i in range(2, 14+1):
    l65_to_hb[('nahum', 2, i)]=('nahum', 2, i-1)

#nehemiah
l65_to_hb[('nehemiah', 10, 1)] = ('nehemiah', 9, 38)
for i in range(2, 40+1):
    l65_to_hb[('nehemiah', 10, i)]=('nehemiah', 10, i-1)


#numbers
for i in range(36, 50+1):
    l65_to_hb[('numbers', 17, i-35)]=('numbers', 16, i)
for i in range(16, 28+1):
    l65_to_hb[('numbers', 17, i)]=('numbers', 17, i-15)

l65_to_hb[('numbers', 30, 1)] = ('numbers', 29, 40)
for i in range(2, 17+1):
    l65_to_hb[('numbers', 30, i)]=('numbers', 30, i-1)

#psalms
def add_psalms_merging_verse_1_in_1965(chap, maxverse):
    l65_to_hb_merge[(('psalms', chap, 1),('psalms', chap, 2))] = ('psalms', chap, 1)
    for i in range(3, maxverse+1):
        l65_to_hb[('psalms', chap, i)]=('psalms', chap, i-1)

add_psalms_merging_verse_1_in_1965(3, 9)
add_psalms_merging_verse_1_in_1965(4, 9)
add_psalms_merging_verse_1_in_1965(5, 13)
add_psalms_merging_verse_1_in_1965(6, 11)
add_psalms_merging_verse_1_in_1965(7, 18)
add_psalms_merging_verse_1_in_1965(8, 10)
add_psalms_merging_verse_1_in_1965(9, 21)
add_psalms_merging_verse_1_in_1965(11, 8)
add_psalms_merging_verse_1_in_1965(12, 9)
add_psalms_merging_verse_1_in_1965(13, 7)
add_psalms_merging_verse_1_in_1965(18, 51)
add_psalms_merging_verse_1_in_1965(19, 15)
add_psalms_merging_verse_1_in_1965(20, 10)
add_psalms_merging_verse_1_in_1965(21, 14)
add_psalms_merging_verse_1_in_1965(22, 32)
add_psalms_merging_verse_1_in_1965(30, 13)
#so far ok, therefore will leave the rest of psalms as is no check
add_psalms_merging_verse_1_in_1965(31, 25)
add_psalms_merging_verse_1_in_1965(34, 23)
add_psalms_merging_verse_1_in_1965(36, 13)
add_psalms_merging_verse_1_in_1965(38, 23)
add_psalms_merging_verse_1_in_1965(39, 14)
add_psalms_merging_verse_1_in_1965(40, 18)
add_psalms_merging_verse_1_in_1965(41, 14)
add_psalms_merging_verse_1_in_1965(42, 12)
add_psalms_merging_verse_1_in_1965(44, 27)
add_psalms_merging_verse_1_in_1965(45, 18)
add_psalms_merging_verse_1_in_1965(46, 12)
add_psalms_merging_verse_1_in_1965(47, 10)
add_psalms_merging_verse_1_in_1965(48, 15)
add_psalms_merging_verse_1_in_1965(49, 21)
add_psalms_merging_verse_1_in_1965(51, 20)
add_psalms_merging_verse_1_in_1965(51, 21)
add_psalms_merging_verse_1_in_1965(52, 10)
add_psalms_merging_verse_1_in_1965(52, 11)
add_psalms_merging_verse_1_in_1965(53, 7)
add_psalms_merging_verse_1_in_1965(54, 8)
add_psalms_merging_verse_1_in_1965(54, 9)
add_psalms_merging_verse_1_in_1965(55, 24)
add_psalms_merging_verse_1_in_1965(56, 14)
add_psalms_merging_verse_1_in_1965(57, 12)
add_psalms_merging_verse_1_in_1965(58, 12)
add_psalms_merging_verse_1_in_1965(59, 18)
add_psalms_merging_verse_1_in_1965(60, 13)
add_psalms_merging_verse_1_in_1965(60, 14)
add_psalms_merging_verse_1_in_1965(61, 9)
add_psalms_merging_verse_1_in_1965(62, 13)
add_psalms_merging_verse_1_in_1965(63, 12)
add_psalms_merging_verse_1_in_1965(64, 11)
add_psalms_merging_verse_1_in_1965(65, 14)
add_psalms_merging_verse_1_in_1965(67, 8)
add_psalms_merging_verse_1_in_1965(68, 36)
add_psalms_merging_verse_1_in_1965(69, 37)
add_psalms_merging_verse_1_in_1965(70, 6)
add_psalms_merging_verse_1_in_1965(75, 11)
add_psalms_merging_verse_1_in_1965(76, 13)
add_psalms_merging_verse_1_in_1965(77, 21)
add_psalms_merging_verse_1_in_1965(80, 20)
add_psalms_merging_verse_1_in_1965(81, 17)
add_psalms_merging_verse_1_in_1965(83, 19)
add_psalms_merging_verse_1_in_1965(84, 13)
add_psalms_merging_verse_1_in_1965(85, 14)
add_psalms_merging_verse_1_in_1965(88, 19)
add_psalms_merging_verse_1_in_1965(89, 53)
add_psalms_merging_verse_1_in_1965(92, 16)
add_psalms_merging_verse_1_in_1965(102, 29)
add_psalms_merging_verse_1_in_1965(108, 14)
add_psalms_merging_verse_1_in_1965(140, 14)
add_psalms_merging_verse_1_in_1965(142, 8)

#song
l65_to_hb_merge[(('songs', 4, 16),('songs', 4, 17))] = ('songs', 4, 16)
l65_to_hb[('songs', 7, 1)] = ('songs', 6, 13)
for i in range(2, 14+1):
    l65_to_hb[('songs', 7, i)]=('songs', 7, i-1)

#zechariah
for i in range(18, 21+1):
    l65_to_hb[('zechariah', 2, i-17)]=('zechariah', 1, i)
for i in range(5, 17+1):
    l65_to_hb[('zechariah', 2, i)]=('zechariah', 2, i-4)
#found out..
l65_to_hb_merge[(('genesis', 14, 20),('genesis', 14, 21))] = ('genesis', 24, 20)
l65_to_hb[('genesis', 14, 22)]=('genesis', 14, 21)
l65_to_hb_merge[(('genesis', 14, 23), "zeme")]=(('genesis', 14, 22), ('genesis', 14, 23))


# ebreju vietturis : lv vietturis
hb_to_l65 = {v: k for k, v in l65_to_hb.items() if v[0]!= '!'}
hb_to_l65_merge = {v: k for k, v in l65_to_hb_merge.items()}
for k, v in l65_to_hb_merge.items():
    #multiple hebrews to single latvian
    if isinstance(v[1], tuple):
        for resolvedVerse in v:
            #this is case where split string is specified for single latvian verse
            if(isinstance(k[0], tuple) and isinstance(k[1], str)):
                hb_to_l65[resolvedVerse] = ("!"+k[0][0], k[0][1], k[0][2])
                l65_to_hb[k[0]] = ("!", -1, -1)
            #also never happens in current list, what was the reasoning putting else here, "just so that it is"?
            else:
                hb_to_l65[resolvedVerse] = ("!"+k[0], k[1], k[2])
                l65_to_hb[k[0]] = ("!", -1, -1)
#value is single hebrew verse. so if key 2nd is tuple not string then
    elif isinstance(k[1], tuple):
        #where to split the hebrew verse (no case actually in current list!
        if(isinstance(v[0], tuple) and isinstance(v[1], str)):
            for resolvedVerse in k:
                hb_to_l65[resolvedVerse] = ("!"+v[0][0], v[0][1], v[0][2])
                hb_to_l65[v[0]] = ("!", -2, -2)
        #single heb verse maps to multiple lvs, so put pointer mark ! (watch merges instead of direct)
        #and the rest two just placeholders so that they are not misused anywhere as misguiding references
        else:
            for resolvedVerse in k:
                l65_to_hb[resolvedVerse] = ("!"+v[0], v[1], v[2])
            #print(v[0])
            hb_to_l65[v] = ("!", -2, -2)
### false alarm, this turns out was standard case of 2 hb verses in single lv verse.
#    elif isinstance(k[1], str) and len(v) == 3:
#        #"part of the verse is one verse in hebrew". for now i will leave the latvian to hebrew map not specified, as its not used for now anyways
#        hb_to_l65[v] = ("!", -2, -2)
#        #print(v)
#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!! IZLAISTS PANTS 1965
#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
hb_to_l65[('exodus', 1, 21)] = None

In [17]:
lxx_to_hb = {}
# Greek verse splits: one LXX verse covers multiple Hebrew verses
# Format: lxx_to_hb_merge[(lxx_key, split_token)] = (hb_key1, hb_key2, ...)
# split_token = last word included in the first part (same convention as l65_to_hb_merge)
lxx_to_hb_merge = {}

# Hebrew 21:1 maps to the tail of LXX 20:42
#1 samuel 20:42+
lxx_to_hb_merge[('1_samuel', 20, 42), "αἰῶνος"]=(
    ('1_samuel', 20, 42),
    ('1_samuel', 21, 1)
)# merged into end of 20:42
for i in range(1, 13+1):
    lxx_to_hb[('1_samuel', 21, i)]=('1_samuel', 21, i+1)
lxx_to_hb_merge[(('1_samuel', 21, 14), ('1_samuel', 21, 15))] = ('1_samuel', 21, 15)

# 1 Chronicles 12
lxx_to_hb_merge[(('1_chronicles', 12, 4), "τῶν τριάκοντα")] = (
    ('1_chronicles', 12, 4), ('1_chronicles', 12, 5)
)
for i in range(6, 40+1):
    lxx_to_hb[('1_chronicles', 12, i-1)] = ('1_chronicles', 12, i)
lxx_to_hb_merge [(('1_chronicles', 12, 39), ('1_chronicles', 12, 40))] = ('1_chronicles', 12, 40)



In [18]:
# ebreju vietturis : lxx vietturis
hb_to_lxx = {v: k for k, v in lxx_to_hb.items() if v[0]!= '!'}
hb_to_lxx_merge = {v: k for k, v in lxx_to_hb_merge.items()}
for k, v in lxx_to_hb_merge.items():
    #multiple hebrews to single latvian
    if isinstance(v[1], tuple):
        for resolvedVerse in v:
            #this is case where split string is specified for single latvian verse
            if(isinstance(k[0], tuple) and isinstance(k[1], str)):
                hb_to_lxx[resolvedVerse] = ("!"+k[0][0], k[0][1], k[0][2])
                lxx_to_hb[k[0]] = ("!", -1, -1)
            #also never happens in current list, what was the reasoning putting else here, "just so that it is"?
            else:
                hb_to_lxx[resolvedVerse] = ("!"+k[0], k[1], k[2])
                lxx_to_hb[k[0]] = ("!", -1, -1)
#value is single hebrew verse. so if key 2nd is tuple not string then
    elif isinstance(k[1], tuple):
        #where to split the hebrew verse (no case actually in current list!
        if(isinstance(v[0], tuple) and isinstance(v[1], str)):
            for resolvedVerse in k:
                hb_to_lxx[resolvedVerse] = ("!"+v[0][0], v[0][1], v[0][2])
                hb_to_lxx[v[0]] = ("!", -2, -2)
        #single heb verse maps to multiple lvs, so put pointer mark ! (watch merges instead of direct)
        #and the rest two just placeholders so that they are not misused anywhere as misguiding references
        else:
            for resolvedVerse in k:
                lxx_to_hb[resolvedVerse] = ("!"+v[0], v[1], v[2])
            #print(v[0])
            hb_to_lxx[v] = ("!", -2, -2)

In [19]:
def split_str_tokened(input_string, tokens):
    result = []
    current_string = input_string

    for token in tokens:
        # Atrodam pirmo marķiera parādīšanās vietu
        token_index = current_string.find(token)
        #print(token_index)
        #print(f"::str:: {current_string} \n::token::  {token}")
        # Ja marķieris netiek atrasts, pārtraucam procesu
        if token_index == -1:
            break

        # Sadalām virkni:
        # pirmā daļa līdz marķiera beigām (ieskaitot pašu marķieri)
        # otrā daļa - viss, kas paliek aiz marķiera
        split_pos = token_index + len(token)
        part = current_string[:split_pos]
        current_string = current_string[split_pos:].strip()

        result.append(part)


    if current_string:
        result.append(current_string)
    #print(result)
    #print("-"*50)
    #print(tokens)
    return result

In [20]:
def get_greek_text_for_hb_key(key, strongs_g, back_g):
    """
    Given a Hebrew MT verse key, return the LXX DataFrame for that verse.
    Handles splits: when one LXX verse covers multiple Hebrew verses,
    uses hb_to_lxx_merge to split and return only the relevant portion.
    Returns a sorted DataFrame with 'form' column, or None if not available.
    """
    #lxx_key, status = resolve_lxx_key(hb_key)

    # find the full DF for the LXX verse
    #full_df = None

    lxx_text=""
    if key in hb_to_lxx.keys():
        kkey = hb_to_lxx[key]
        if(kkey == None):
            lxx_text="-"
        else:
            if not kkey[0][0]=="!":
                lxx_text=" ".join(strongs_g.get_group(kkey).sort_values("word")["form"].astype(str))
            else:
                nkey = (kkey[0][1:], kkey[1], kkey[2])
                #print("here")
                #print(nkey)
                for k, v in lxx_to_hb_merge.items():
                    #merge will ALWAY have at least 2 items in key (either mergable verse Nr2 or splitstring Nr1)
                    if(isinstance(k[1], str) and k[0] == nkey):

                        #case 1: lv verse has multiple hebrw verses in it
                        sstr = " ".join(strongs_g.get_group(nkey).sort_values("word")["form"].astype(str))
                        spArr = split_str_tokened(sstr, [k[i] for i in range(1, len(k))])
                        #print(spArr)
                        if(len(spArr) != len(v)):
                            raise Exception (f"could not tokenise properly!!!@ {k} to {v}")
                        for i, vsNm in enumerate(v):
                            if vsNm == key:
                                lxx_text=spArr[i]
                                break
                        if not lxx_text:
                            raise Exception (f"could not find verse part for the {key} at {k} to {v}")
                    elif(v==key):
                        #now this is inverse case of the above, lxx verses should be merged.
                        rslt = []
                        for mrg_verse in k:
                            rslt.append(" ".join(strongs_g.get_group(mrg_verse).sort_values("word")["form"].astype(str)))
                        lxx_text = " ".join(rslt)
                        break
                if not  lxx_text:
                    raise Exception (f"key not found! {nkey} :: {key}")
    else:
        if key in strongs_g.groups:
            lxx_text=" ".join(strongs_g.get_group(key).sort_values("word")["form"].astype(str))
        else:
            if key in back_g.groups:
                lxx_text=" ".join(back_g.get_group(key).sort_values("word_idx")["form"].astype(str))
            else:
                lxx_text=""
    return lxx_text


def get_greek_df_for_hb_key(key, strongs_g, back_g):
    """
    Given a Hebrew MT verse key, return the LXX DataFrame for that verse.
    Handles splits: when one LXX verse covers multiple Hebrew verses,
    uses hb_to_lxx_merge to split and return only the relevant portion.
    Returns a sorted DataFrame with 'form' column, or None if not available.
    """
    #lxx_key, status = resolve_lxx_key(hb_key)

    # find the full DF for the LXX verse
    #full_df = None

    lxx_text= strongs_g.obj.iloc[0:0]
    if key in hb_to_lxx.keys():
        kkey = hb_to_lxx[key]
        if(kkey == None):
            lxx_text=strongs_g.obj.iloc[0:0]
        else:
            if not kkey[0][0]=="!":
                return strongs_g.get_group(kkey).sort_values("word")
            else:
                nkey = (kkey[0][1:], kkey[1], kkey[2])
                #print("here")
                #print(nkey)
                for k, v in lxx_to_hb_merge.items():
                    #merge will ALWAY have at least 2 items in key (either mergable verse Nr2 or splitstring Nr1)
                    if(isinstance(k[1], str) and k[0] == nkey):
                        #case 1: lv verse has multiple hebrw verses in it
                        full_df = strongs_g.get_group(nkey).sort_values("word")
                        merge_key=k
                        merge_lxx_key = merge_key[0]
                        split_tokens = [merge_key[i] for i in range(1, len(merge_key))]
                        # this LXX verse is split — find which part belongs to hb_key (key variable)
                        full_text = " ".join(full_df["form"].astype(str))
                        parts = split_str_tokened(full_text, split_tokens)
                        if len(parts) != len(v):
                            print(f"  ⚠️🇬🇷 LXX split failed for {lxx_key}: got {len(parts)} parts, expected {len(v)}")
                            return full_df  # fallback: return full verse
                        #key is hebrew
                        part_idx = v.index(key)
                        part_text = parts[part_idx]
                        # figure out which rows of the DF belong to this part
                        # by counting words consumed by earlier parts
                        words_before = sum(len(p.split()) for p in parts[:part_idx])
                        words_in_part = len(part_text.split())
                        return full_df.iloc[words_before : words_before + words_in_part]
                    elif(v==key):
                        #now this is inverse case of the above, lxx verses should be merged.
                        merged_dfs = []
                        for mrg_lxx_verse in k:
                            if mrg_lxx_verse in strongs_g.groups:
                                merged_dfs.append(strongs_g.get_group(mrg_lxx_verse).sort_values("word"))
                        if merged_dfs:
                            return pd.concat(merged_dfs, ignore_index=True)
                if not  lxx_text:
                    raise Exception (f"key not found! {nkey} :: {key}")
    else:
        if key in strongs_g.groups:
            return strongs_g.get_group(key).sort_values("word")
        else:
            if key in back_g.groups:
                return back_g.groupd.get_group(key).sort_values("word_idx")
            else:
                return strongs_g.obj.iloc[0:0]


In [21]:
def get_l65_text_for_hb_key(key, l65_g):
    l65_text=""
    if key in hb_to_l65.keys():
        kkey = hb_to_l65[key]
        if(kkey == None):
            l65_text="-"
        else:
            #print(kkey)
            if not kkey[0][0]=="!":
                l65_text=" ".join(l65_g.get_group(kkey).sort_values("word_idx")["form"].astype(str))
            else:
                nkey = (kkey[0][1:], kkey[1], kkey[2])
                #print("here")
                #print(nkey)
                for k, v in l65_to_hb_merge.items():
                    #merge will ALWAY have at least 2 items in key (either mergable verse Nr2 or splitstring Nr1)
                    if(isinstance(k[1], str) and k[0] == nkey):

                        #case 1: lv verse has multiple hebrw verses in it
                        sstr = " ".join(l65_g.get_group(nkey).sort_values("word_idx")["form"].astype(str))
                        spArr = split_str_tokened(sstr, [k[i] for i in range(1, len(k))])
                        #print(spArr)
                        if(len(spArr) != len(v)):
                            raise Exception (f"could not tokenise properly!!!@ {k} to {v}")
                        for i, vsNm in enumerate(v):
                            if vsNm == key:
                                l65_text=spArr[i]
                                break
                        if not l65_text:
                            raise Exception (f"could not find verse part for the {key} at {k} to {v}")
                    elif(v==key):
                        #now this is inverse case of the above, l65 verses should be merged.
                        rslt = []
                        for mrg_verse in k:
                            rslt.append(" ".join(l65_g.get_group(mrg_verse).sort_values("word_idx")["form"].astype(str)))
                        l65_text = " ".join(rslt)
                        break
                if not  l65_text:
                    raise Exception (f"key not found! {nkey} :: {key}")
    else:
        l65_text=" ".join(l65_g.get_group(key).sort_values("word_idx")["form"].astype(str))
                # 2. Validation: Hebrew Word Count and Order
    return l65_text



def get_hb_keyed_text_for_l65_key(key, hb_keyed_g, sort_col='word'):
    """Mirror of get_l65_text_for_hb_key: given an l65-numbered verse key,
    return the text from an hb-numbered df (hb_df / strongs / abp / 1694).
    Handles splits and merges using l65_to_hb / hb_to_l65_merge."""
    text = ""
    if key in l65_to_hb.keys():
        kkey = l65_to_hb[key]
        if kkey is None:
            text = "-"
        else:
            if not kkey[0][0] == "!":
                text = " ".join(hb_keyed_g.get_group(kkey).sort_values(sort_col)["form"].astype(str))
            else:
                nkey = (kkey[0][1:], kkey[1], kkey[2])
                for k, v in hb_to_l65_merge.items():
                    # merge will ALWAYS have at least 2 items in key
                    if isinstance(k[1], str) and k[0] == nkey:
                        # case 1: hb verse has multiple l65 verses in it -> split by tokens
                        sstr = " ".join(hb_keyed_g.get_group(nkey).sort_values(sort_col)["form"].astype(str))
                        spArr = split_str_tokened(sstr, [k[i] for i in range(1, len(k))])
                        if len(spArr) != len(v):
                            raise Exception(f"could not tokenise properly!!!@ {k} to {v}")
                        for i, vsNm in enumerate(v):
                            if vsNm == key:
                                text = spArr[i]
                                break
                        if not text:
                            raise Exception(f"could not find verse part for the {key} at {k} to {v}")
                    elif v == key:
                        # inverse: multiple hb verses should be merged into this single l65 verse
                        rslt = []
                        for mrg_verse in k:
                            rslt.append(" ".join(hb_keyed_g.get_group(mrg_verse).sort_values(sort_col)["form"].astype(str)))
                        text = " ".join(rslt)
                        break
                if not text:
                    raise Exception(f"key not found! {nkey} :: {key}")
    else:
        text = " ".join(hb_keyed_g.get_group(key).sort_values(sort_col)["form"].astype(str))
    return text


## darba lauks

In [22]:
nt_df.head()

,book,chapter,verse,word,greek,latvian
0,1_corinthians,1,1,0,Παῦλος,[Pāvils]
1,1_corinthians,1,1,1,κλητὸς,[aicināts]
2,1_corinthians,1,1,2,ἀπόστολος,[apustulis]
3,1_corinthians,1,1,3,Χριστοῦ,[Kristus]
4,1_corinthians,1,1,4,Ἰησοῦ,[Jēzus]


In [23]:
ot_df.head()

,book,chapter,verse,word,hebrew,greek,latvian
0,1_chronicles,1,1,0,אָדָ֥ם,[Αδαμ],[Ādams]
1,1_chronicles,1,1,1,שֵׁ֖ת,[Σηθ],[Sets]
2,1_chronicles,1,1,2,אֱנֽוֹשׁ׃,[Ενως],[Ēnošs]
3,1_chronicles,1,2,0,קֵינָ֥ן,[Καιναν],[Kenans]
4,1_chronicles,1,2,1,מַהֲלַלְאֵ֖ל,[Μαλελεηλ],[Mahalaleēls]


### 1.1. visvienkāršākais, esošs meklējums 1 valodas bībeles konkrētajā izdevumā

In [24]:
import unicodedata
def raccs_common(text):
    d = {
        ord('\u2019'): None,  # RIGHT SINGLE QUOTATION MARK  \u2019
        ord('\u2018'): None,  # LEFT SINGLE QUOTATION MARK   \u2018
        ord('\u201C'): None,  # LEFT DOUBLE QUOTATION MARK
        ord('\u201D'): None,  # RIGHT DOUBLE QUOTATION MARK
        ord('['): None,
        ord(']'): None,
        ord('-'): None,
        ord("'"): None,
        ord('\u29FC'): None,  # \u29fc
        ord('\u29FD'): None,  # \u29fd
        ord('*'): None,
        ord('\u21D4'): None,  # \u21d4
        ord('\u3009'): None,  # \u3009
        ord('\u3008'): None,  # \u3008
        ord('\u203F'): None,  # \u203f
        ord('\u00AB'): None,  # \u00ab
        ord('\u00BB'): None,  # \u00bb
        ord('\u2039'): None,  # \u2039
        ord('\u203A'): None,  # \u203a
        ord('('): None,
        ord(')'): None,
        ord(';'): None,
        ord('{'): None,
        ord('}'): None,
        ord('.'): None,
        ord(','): None,
        ord('!'): None,
        ord('?'): None,
        ord(';'): None,
        ord(':'): None,
        ord('"'): None,
        ord(')'): None,
        ord('’'): None,  # RIGHT SINGLE QUOTATION MARK
        ord('‘'): None,  # LEFT SINGLE QUOTATION MARK
        ord('“'): None,  # LEFT DOUBLE QUOTATION MARK
        ord('”'): None,  # RIGHT DOUBLE QUOTATION MARK
        ord('['): None,  # LEFT SINGLE QUOTATION MARK
        ord(']'): None,  # RIGHT SINGLE QUOTATION MARK
        ord('-'): None,   # HYPHEN-MINUS
        ord('⧼'): None,  # LEFT WHITE ANGLE BRACKET
        ord('⧽'): None,  # RIGHT WHITE ANGLE BRACKET
        ord('*'): None,   # ASTERISK
        ord('⇔'): None,  # LEFT RIGHT DOUBLE ARROW
        ord('〉'): None,  # GREATER-THAN SIGN
        ord('〈'): None,  # LESS-THAN SIGN
        ord('‿'): None,  # LOW LINE
        ord('«'): None,  # LEFT-POINTING DOUBLE ANGLE QUOTATION MARK
        ord('»'): None,  # RIGHT-POINTING DOUBLE ANGLE QUOTATION MARK
        ord('‹'): None,  # SINGLE LEFT-POINTING ANGLE QUOTATION MARK
        ord('›'): None,  # SINGLE RIGHT-POINTING ANGLE QUOTATION MARK
        ord('('): None,  # LEFT PARENTHESIS
        ord(')'): None,  # RIGHT PARENTHESIS
# nestrādā .... nooooooooooooooooooooooo
        ord('-') : None,  # HYPHEN-MINUS
        ord(';') : None,  # SEMICOLON
        ord('᾽') : None,  # COLON
    }
    return unicodedata.normalize('NFC', text).translate(d)

def strip_greek_diacritics(s):
    return raccs_common(''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn'))

In [25]:

SEARCH1='μορφ'

strongs_df
#nt_strongs_df
abp_strongs_df

atgKaSauc = """
l65_df
l24_df
hb_df
strongs_df
abp_strongs_df
l1694_df
"""
#, case=False, regex=False
gk_ic_lxx_tr = searchExecutorSimple(strongs_df, strip_greek_diacritics, case=False, regex=False)
gk_ic_abp_tr = searchExecutorSimple(abp_strongs_df, strip_greek_diacritics, case=False, regex=False)
gk_ic_un_tr = searchExecutorSimple(nt_strongs_df, strip_greek_diacritics, case=False, regex=False)

resultats_morf_lxx_1 = gk_ic_lxx_tr.search(SEARCH1)
print(resultats_morf_lxx_1.head())

resultats_morf_abp_1 = gk_ic_abp_tr.search(SEARCH1)
print(resultats_morf_abp_1.head())

resultats_morf_un_nt_1 = gk_ic_un_tr.search(SEARCH1)
print(resultats_morf_un_nt_1.head())

          form strong_num strong_en_title    book  chapter  verse  word  \
144783   μορφὴ       3444           N-NSF  judges        8     18    23   
288811   μορφὴ       3444           N-NSF     job        4     16     8   
370916  μορφὴν       3444           N-ASF  isaiah       44     13    14   
444489   μορφή       3444           N-NSF  daniel        4     36    19   
444702   μορφὴ       3444           N-NSF  daniel        5      6     4   

       is_patched form_en strong_title  
144783        NaN     NaN          NaN  
288811        NaN     NaN          NaN  
370916        NaN     NaN          NaN  
444489        NaN     NaN          NaN  
444702        NaN     NaN          NaN  
        verse  word_idx         form             form_en strong_num  \
337417     13         8    εμόρφωσεν               forms       3445   
337434     13        25       μορφήν      the appearance       3444   
394412     16         7        μορφή  1there] appearance       3444   
438553      6      

In [26]:
len(resultats_morf_abp_1)

8

In [27]:
strip_greek_diacritics("ά'αβγδεζηθικλμνξοπρςστυφχψω")

'ααβγδεζηθικλμνξοπρςστυφχψω'

In [28]:
l1694_df.query(
    "book=='genesis' & chapter==15 & verse==6"
)

,book,chapter,verse,word_idx,form
173448,genesis,15,6,0,Un
173449,genesis,15,6,1,wiꞥſch
173450,genesis,15,6,2,tizzeja
173451,genesis,15,6,3,eekẜch
173452,genesis,15,6,4,to
173453,genesis,15,6,5,KUNGU
173454,genesis,15,6,6,un
173455,genesis,15,6,7,tas
173456,genesis,15,6,8,peelahgadija
173457,genesis,15,6,9,wiꞥꞥam


In [29]:
resultats_morf_un_nt_1.query(
    "book=='2_corinthians' & chapter==3 & verse==18"
)

,verse,word,form,form_en,strong_num,strong_title,strong_en_title,translit,translit_title,book,chapter
16336,18,12,μεταμορφούμεθα,are being transformed into,3339,"3339: To transform, transfigure. From meta and...",V-PIM/P-1P,metamorphoumetha,"metamorphoumetha: To transform, transfigure. F...",2_corinthians,3


#### 1.1.1. salīdzinājums

In [30]:
def printDiffCounts(set1, set2, titleString='',
                    firstName='1.', secondName='2.'):
  only_in_1 =  set1 - set2
  only_in_2 =  set2 - set1
  in_both = set1 & set2
  print(f"{titleString} skaiti: kopējie: {len(in_both)}\n {firstName}: {len(only_in_1)}\n {secondName}: {len(only_in_2)}\n")
  print(f"{titleString} SUM: {firstName}: {len(set1)} {secondName}: {len(set2)}")


In [31]:
set_lxx_morf_1 = verse_set(resultats_morf_lxx_1)
set_abp_morf_1 = verse_set(resultats_morf_abp_1)

In [32]:
printDiffCounts(set_lxx_morf_1, set_abp_morf_1,
                SEARCH1.capitalize(), 'LXX', 'ABP')

Μορφ skaiti: kopējie: 7
 LXX: 1
 ABP: 0

Μορφ SUM: LXX: 8 ABP: 7


In [33]:
SEARCH2='γαπ'
SEARCH3='στοργ'
resultats_γαπ_lxx_2 = gk_ic_lxx_tr.search(SEARCH2)
print(resultats_γαπ_lxx_2[:3])

resultats_γαπ_abp_2 = gk_ic_abp_tr.search(SEARCH2)
print(resultats_γαπ_abp_2[:3])

resultats_γαπ_un_2 = gk_ic_un_tr.search(SEARCH2)
print(resultats_γαπ_un_2[:3])

           form strong_num strong_en_title     book  chapter  verse  word  \
11353  ἀγαπητόν         27         Adj-ASM  genesis       22      2     7   
11355  ἠγάπησας         25        V-AAI-2S  genesis       22      2     9   
11618  ἀγαπητοῦ         27         Adj-GSM  genesis       22     12    29   

      is_patched form_en strong_title  
11353        NaN     NaN          NaN  
11355        NaN     NaN          NaN  
11618        NaN     NaN          NaN  
       verse  word_idx    form   form_en strong_num    strong_title      book  \
2870       5        16  αγαπάν   to love         25         to love    joshua   
8696      11         6  αγαπάν   to love         25         to love    joshua   
18810      2        15  αγάπης  the love         26  love, goodwill  jeremiah   

       chapter  
2870        22  
8696        23  
18810        2  
      verse  word      form  form_en  strong_num  \
698      14     1  ἀγαπητοί  beloved          27   
1987      1     9    ἀγάπην     lo

In [34]:
set_γαπ_lxx_2 = verse_set(resultats_γαπ_lxx_2)
set_γαπ_abp_2 = verse_set(resultats_γαπ_abp_2)
printDiffCounts(set_γαπ_lxx_2, set_γαπ_abp_2,
                SEARCH2, 'LXX', 'ABP')

γαπ skaiti: kopējie: 227
 LXX: 5
 ABP: 3

γαπ SUM: LXX: 232 ABP: 230


In [35]:
" ".join(l1694_df.query(
    "book=='genesis' & chapter==8 & verse==1"
)["form"])

'UN DEews atgahdajahs Noßs un wiẜẜo Swehru un wiẜẜo lohpu kas ar to Ꞩchꞣirſtâ bija un DEews dewe Wehju wirs Semmes un tee Uhdeni norimmahs'

at**cer**ējās -> at**gād**ājās

In [36]:
no=5
lidz=15
print(resultats_γαπ_lxx_2[["book", "chapter", "verse", "word", "strong_num", "form"]][no:lidz+1])
print(resultats_γαπ_abp_2[["book", "chapter", "verse", "word_idx", "strong_num", "form"]][no:lidz+1])


          book  chapter  verse  word  strong_num      form
14344  genesis       25     28     0          25  ἠγάπησεν
14357  genesis       25     28    13          25     ἠγάπα
17156  genesis       29     18     0          25  ἠγάπησεν
17208  genesis       29     20    16          25    ἀγαπᾶν
17373  genesis       29     30     4          25  ἠγάπησεν
17422  genesis       29     32    23          25  ἀγαπήσει
20829  genesis       34      3     9          25  ἠγάπησεν
22815  genesis       37      3     2          25     ἠγάπα
28436  genesis       44     20    29          25  ἠγάπησεν
44448   exodus       20      6     6          25  ἀγαπῶσίν
44934   exodus       21      5     6  3756|665.1   ἠγάπηκα
           book  chapter  verse  word_idx strong_num        form
22809  jeremiah       14     10         5         25    ηγάπησαν
25800  jeremiah        5     31        13         25    ηγάπησεν
29939  jeremiah       11     15         2         25   ηγαπημένη
30722  jeremiah       31      3 

In [37]:
" ".join(l1694_df.query(
    "book=='genesis' & chapter==49 & verse==18"
)["form"])

'KUNGS es gaidu us tawu Peſtiẜchanu'

In [38]:
SEARCH3='στοργ'
resultats_γαπ_lxx_3 = gk_ic_lxx_tr.search(SEARCH3)
print(resultats_γαπ_lxx_3[:3])

resultats_γαπ_abp_3 = gk_ic_abp_tr.search(SEARCH3)
print(resultats_γαπ_abp_3[:3])

resultats_γαπ_un_3 = gk_ic_un_tr.search(SEARCH3)


Empty DataFrame
Columns: [form, strong_num, strong_en_title, book, chapter, verse, word, is_patched, form_en, strong_title]
Index: []
Empty DataFrame
Columns: [verse, word_idx, form, form_en, strong_num, strong_title, book, chapter]
Index: []


In [39]:
len(resultats_γαπ_un_3)

3

In [40]:
print(resultats_γαπ_un_3[["book", "chapter", "verse", "word", "form", "form_en", "strong_num", "strong_en_title", "translit", "strong_title"]][:5])

             book  chapter  verse  word         form    form_en  strong_num  \
21168   2_timothy        3      3     0     ἄστοργοι   unloving         794   
131678     romans        1     31     2    ἀστόργους  heartless         794   
132815     romans       12     10     4  φιλόστοργοι    devoted        5387   

       strong_en_title      translit  \
21168          Adj-NMP      astorgoi   
131678         Adj-AMP     astorgous   
132815         Adj-NMP  philostorgoi   

                                             strong_title  
21168   794: Unloving, devoid of affection. Hard-heart...  
131678  794: Unloving, devoid of affection. Hard-heart...  
132815  5387: From philos and storge; fond of natural ...  


## 2. vizualizācija

**Ideja (S)**: vizualizēt (visas) vietas, atšķirīgās no kopīgajām atsevišķi (3 skati uz katru no pāriem ik meklējumā): <br><br>
piemēram, pārim 65, gliks:
<br><br>
mīl: (65, gliks, kopīgie)| (65,gliks, tikai gliks) | (65, gliks, tikai 65) <br><br>
tic: (65, gliks, kopīgie)| (65,gliks, tikai gliks) | (65, gliks, tikai 65)utt.

### 2.1. def

In [41]:
def download_txt_from_private_repo(repopath, path_in_repo, token, *args, **kwargs):
	url = f"https://api.github.com/repos/{repopath}/contents/{path_in_repo}"
	r = requests.get(url, headers={
		"Authorization": f"token {token}",
		"Accept": "application/vnd.github.v3.raw"
	})
	r.raise_for_status()
	return r.text


In [42]:
MANIFEST_PATH_GREEK="mp3list_g.csv"
MANIFEST_PATH_HEBREW="mp3list_h.csv"
OT_REPO_MANIFEST_PATH = "ewilpeers/bible"
TOKEN_OT_MANF_FU_GITHUB = b'\xb3\xb4\x3a\x34\xba\x31\xaf\x38\xb0\x3a\xaf\x98\x98\x21\x29\x9a\x23\xa2\x23\xa4\x18\x25\xba\x25\xba\xb8\xa3\xba\xbc\x9c\xb5\x3c\xa9\xaf\x38\xa0\x33\x28\xa8\x23\x18\x18\xb8\x37\x3b\xa2\x21\x19\xa9\xb1\xa7\xb5\xb0\x31\x3c\x31\x31\x3c\xac\xb0\xb4\x1a\xbb\x39\x19\xb5\x32\x1b\x18\x23\xb1\x3b\xb8\x3c\x35\x22\x27\x2b\xa3\x22\x2b\xa4\xab\xaa\x1a\x31\xb6\x1a\x31\xbb\x9a\x24\x1c'
TOKEN_OT_MANF = gh.decode(TOKEN_OT_MANF_FU_GITHUB)
if(not os.path.exists(MANIFEST_PATH_GREEK)):
  with open(MANIFEST_PATH_GREEK, 'w') as f:
    f.write(download_txt_from_private_repo(OT_REPO_MANIFEST_PATH, f"ci/{MANIFEST_PATH_GREEK}", TOKEN_OT_MANF))

if(not os.path.exists(MANIFEST_PATH_HEBREW)):
  with open(MANIFEST_PATH_HEBREW, 'w') as f:
    f.write(download_txt_from_private_repo(OT_REPO_MANIFEST_PATH, f"ci/{MANIFEST_PATH_HEBREW}", TOKEN_OT_MANF))

In [43]:
_MP3_MANIFEST_GREEK = None
_MP3_MANIFEST_HEBREW = None
AUDIO_BASE_URL_GREEK="https://t.noit.pro/strongs_p_g/"
AUDIO_BASE_URL_HEBREW="https://t.noit.pro/strongs_p/"
MANIFEST_PATH_GREEK="mp3list_g.csv"
MANIFEST_PATH_HEBREW="mp3list_h.csv"
from pathlib import Path
def load_mp3_manifest(manifest_path="mp3list_g.csv", language="greek"):
    """Load mp3 manifest from CSV. Falls back to empty set if file missing."""
    global _MP3_MANIFEST_GREEK, _MP3_MANIFEST_HEBREW
    manifest = set()
    if not (_MP3_MANIFEST_GREEK if language == "greek" else _MP3_MANIFEST_HEBREW):
        p = Path(manifest_path)
        if p.exists():
            df = pd.read_csv(p)
            manifest = set(df["filename"].str.strip())
            print(f"  📋 Loaded {len(manifest)} entries from {p}")
        else:
            print(f"  ⚠️ {p} not found — {language} audio players will be disabled")
            manifest = set()
    if language == "greek":
        _MP3_MANIFEST_GREEK = manifest
    else:
        _MP3_MANIFEST_HEBREW = manifest


def make_audio_players(strong_num_raw,
                       v_num,
                       word_idx,
                       prependix="h",
                       subword_idx='',
                       br_prepend_start=True,
                       manifest_path=None):
    """
    Build inline audio play buttons resolved via manifest lookup.

    prependix: "h" for Hebrew (default), "g" for Greek
    manifest_path: override CSV path; defaults per language
    """
    global _MP3_MANIFEST_GREEK, _MP3_MANIFEST_HEBREW

    if prependix == "g":
        if _MP3_MANIFEST_GREEK is None:
            load_mp3_manifest(manifest_path or MANIFEST_PATH_GREEK, language="greek")
        manifest = _MP3_MANIFEST_GREEK
        base_url = AUDIO_BASE_URL_GREEK
    else:
        if _MP3_MANIFEST_HEBREW is None:
            load_mp3_manifest(manifest_path or MANIFEST_PATH_HEBREW, language="hebrew")
        manifest = _MP3_MANIFEST_HEBREW
        base_url = AUDIO_BASE_URL_HEBREW

    if not strong_num_raw:
        return ""
    try:
        sn = int(strong_num_raw)
    except (ValueError, TypeError):
        return ""
    if sn <= 0:
        return ""

    skey = f"{prependix}{sn:04d}"

    variants = []
    if f"{skey}.mp3" in manifest:
        variants.append((f"{base_url}/{skey}.mp3", ""))
    vi = 2
    while f"{skey}-{vi}.mp3" in manifest:
        variants.append((f"{base_url}/{skey}-{vi}.mp3", f" {vi}"))
        vi += 1

    if not variants:
        return ""

    out = "<br>" if br_prepend_start else ""
    for src, label in variants:
        play_label = f"▶{label}"
        out += (
            f'<button data-label="{play_label}" data-suffix="{label}" '
            f'style="font-size:0.8em;padding:1px 5px;cursor:pointer" '
            f"onclick=\"playStrong('{src}', this)\">"
            f'{play_label}</button> '
        )
    return out
def getHead():
      css = """
    <style>
        body { font-family: 'Segoe UI', Tahoma, sans-serif; line-height: 1.6; color: #333; max-width: 1600px; margin: 0 auto; padding: 20px; background-color: #f8f9fa; }
        h1 { text-align: center; color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px; }
        .verse-container { background-color: white; border-radius: 10px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); margin-bottom: 40px; padding: 25px; border-left: 5px solid #3498db; }
        .verse-header { font-weight: bold; color: #2c3e50; background-color: #ecf0f1; padding: 8px 15px; border-radius: 20px; margin-bottom: 15px; }
        .verse-lines { display: flex; flex-wrap: wrap; gap: 15px; margin-bottom: 20px; }
        .line-box { flex: 1 1 45%; min-width: 300px; }
        .line-label { font-weight: bold; color: #16a085; margin-bottom: 5px; }
        .line-content { background-color: #f0f7f4; padding: 12px; border-radius: 8px; border: 1px solid #bdc3c7; font-size: 1.1em; }
        .hebrew-line { background-color: #f0f0f0; font-family: 'Times New Roman', serif; }

        /* Table Styles */
        .mapping-table { width: 100%; border-collapse: collapse; margin-top: 15px; font-size: 0.9em; }
        .mapping-table th { background-color: #3498db; color: white; padding: 12px; text-align: left; position: sticky; top: 0; }
        .mapping-table td { padding: 10px; border-bottom: 1px solid #ddd; vertical-align: top; }

        .hebrew-word { font-weight: bold; color: #8e44ad; font-size: 1.1em; }
        .hebrew-form { color: #7f8c8d; font-weight: normal; font-size: 0.85em; }
        .latvian-word { font-weight: bold; color: #27ae60; }

        /* Morphology Tooltip Style */
        .morph-info {
            font-style: italic;
            color: #e67e22;
            cursor: text;
            border-bottom: 1px dotted #e67e22;
            display: inline-block;
        }
        .definition-cell { color: #555; font-size: 0.85em; line-height: 1.3; max-width: 400px; }
        .index-badge { display: inline-block; width: 25px; height: 25px; background-color: #e74c3c; color: white; border-radius: 50%; text-align: center; line-height: 25px; margin-right: 8px; }
        /* group greek play button(s) together with the word */
        .greek-audio-group {
          display: inline-flex;
          align-items: baseline;
          white-space: nowrap;
        }
/* ⓘ disclosure widget  */
    .verse-info {
        display: inline;
        position: relative;
    }
    .verse-info summary {
        display: inline-flex;
        align-items: center;
        justify-content: center;
        width: 22px;
        height: 22px;
        border-radius: 50%;
        background: #3498db;
        color: white;
        font-size: 13px;
        font-weight: bold;
        cursor: pointer;
        list-style: none;           /* hide default triangle — Firefox */
        margin-left: 8px;
        vertical-align: middle;
        user-select: none;
        transition: background 0.2s;
        line-height: 1;
    }
    .verse-info summary::-webkit-details-marker { display: none; }  /* Chrome/Safari */
    .verse-info summary::marker { content: ''; font-size: 0; }     /* Firefox fallback */
    .verse-info summary:hover,
    .verse-info summary:focus-visible {
        background: #2980b9;
        outline: 2px solid #2980b9;
        outline-offset: 2px;
    }
    .verse-info[open] summary { background: #2c3e50; }

    .verse-info .info-popup {
        display: block;
        margin-top: 10px;
        padding: 10px 14px;
        background: #eaf4fb;
        border: 1px solid #aed6f1;
        border-radius: 8px;
        font-size: 0.95em;
        color: #2c3e50;
        line-height: 1.5;
    }
    .info-popup .info-label {
        font-weight: bold;
        color: #2471a3;
        margin-bottom: 2px;
    }
 .diffV { background-color: #F6AA11; }

        /* gothic old print render */
@font-face {
    font-family: 'UnifrakturMaguntia';
    src: url('../fonts/unifrakturmaguntia-webfont.woff2') format('woff2'),
         url('../fonts/unifrakturmaguntia-webfont.woff') format('woff'),
         url('../fonts/unifrakturmaguntia-webfont.ttf') format('truetype');
    font-weight: normal;
    font-style: normal;
    font-display: swap;
}
.frankfurt-line {
    font-family: 'UnifrakturMaguntia', 'Times New Roman', serif;
   /* background-color: #fdf6ec; */
}
    </style>
    """
      javascript="""
<script type="text/javascript">
var _activeBtn = null;

function playStrong(src, btn) {
  var player = document.getElementById('shared-player');

  // if clicking the same button while playing — stop it
  if (_activeBtn === btn && !player.paused) {
    player.pause();
    player.currentTime = 0;
    btn.textContent = btn.dataset.label;
    _activeBtn = null;
    return;
  }

  // reset previous button if any
  if (_activeBtn) {
    _activeBtn.textContent = _activeBtn.dataset.label;
  }

  _activeBtn = btn;
  btn.textContent = '⏹' + (btn.dataset.suffix || '');
  player.src = src;
  player.play();
}

document.addEventListener('DOMContentLoaded', function() {
  document.getElementById('shared-player').addEventListener('ended', function() {
    if (_activeBtn) {
      _activeBtn.textContent = _activeBtn.dataset.label;
      _activeBtn = null;
    }
  });
});
</script>
    """
      return f"<!DOCTYPE html><html><head><meta charset='UTF-8'>{css}{javascript}</head><body>"

def getTail():
    html += "</body></html>"
    return html
def render_verse(verse_text, needle, transformFun=lambda x: x):
    if not verse_text:
        return ""
    if not needle:
        return verse_text
    t_needle = transformFun(needle)
    if not t_needle:
        return verse_text

    # Build transformed string AND the map: for each char in t_text,
    # record the slice of original indices it came from.
    t_chars = []
    spans   = []   # spans[k] = (orig_start, orig_end) that produced t_chars[k]
    for idx, ch in enumerate(verse_text):
        piece = transformFun(ch)
        for _ in piece:
            t_chars.append(_)
            spans.append((idx, idx + 1))
    t_text = ''.join(t_chars)

    result = []
    cursor = 0          # position in ORIGINAL string
    k = 0               # position in TRANSFORMED string
    n = len(t_needle)
    while k < len(t_text):
        if t_text[k:k+n] == t_needle:
            start = spans[k][0]
            end   = spans[k + n - 1][1]
            # emit any untouched original text between cursor and start
            result.append(verse_text[cursor:start])
            result.append('<span class="diffV">' + verse_text[start:end] + '</span>')
            cursor = end
            k += n
        else:
            k += 1
    result.append(verse_text[cursor:])
    return ''.join(result)

In [44]:
def renderResultsToHTML(results):

    html = getHead()
    for verse_data in results:
        html += chapter_to_html_render(verse_data, html)
    html += getTail()
    return html



### 2.2. action by Claude

In [45]:
# ====== fixes to helpers from 2.1 ======

def getTail():
    return "</body></html>"

def chapter_to_html_render(data):
    """Render a list of verse-dicts belonging to a single book+chapter."""
    if not data:
        return ""
    html = ""
    audio_tag = '<audio id="shared-player"></audio>'
    book_title = data[0].get('book', 'Bible').replace('_', ' ').capitalize()
    chapter = data[0].get('chapter', '')
    html += f"<h1>📖 {book_title} Chapter {chapter}</h1>"
    html += audio_tag

    for verse_data in data:
        v_num = verse_data.get('verse')
        locus = f"{book_title} {chapter}:{v_num}"

        html += '<div class="verse-container">'
        html += f'<div class="verse-header"><span class="index-badge">{v_num}</span> {locus}</div>'

        html += f'''
        <div class="verse-lines">
            <div class="line-box">
                <div class="line-label">🇮🇱 Hebrew:</div>
                <div class="line-content hebrew-line">{verse_data.get('hebrew_text_full_original', '')}</div>
            </div>
            <div class="line-box">
                <div class="line-label">🇱🇻 Latvian (1694):</div>
                <div class="line-content frankfurt-line">{verse_data.get('latvian_text_full_original_1694', '')}</div>
            </div>
        </div>
        '''
        html += f'''
        <div class="verse-lines">
            <div class="line-box">
                <div class="line-label">🇬🇷 Greek LXX:</div>
                <div class="line-content hebrew-line">{verse_data.get('lxx_text_full_original', '')}</div>
                <div class="line-label">🇬🇷 Greek ABP:</div>
                <div class="line-content hebrew-line">{verse_data.get('abp_text_full_original', '')}</div>
            </div>
            <div class="line-box">
                <div class="line-label">🇱🇻 Latvian (1965):</div>
                <div class="line-content">{verse_data.get('latvian_text_full_original_65', '')}
                <div class="verse-info">
                    <div class="info-popup">
                        <div class="info-label">🇱🇻 Latvian (2024):</div>
                        {verse_data.get('latvian_text_full_original_24', '')}
                    </div>
                </div>
                </div>
            </div>
        </div>
        '''
        html += '</div>'
    return html

In [46]:
# ====== 2.2. action — build verse-dicts from the dataframes ======

from functools import lru_cache

# canonical book ordering from ot_df (falls back to alpha if not present)
try:
    _BOOK_ORDER = {b: i for i, b in enumerate(ot_df['book'].drop_duplicates())}
except NameError:
    _BOOK_ORDER = {}

def _sort_key(bcv):
    book, ch, v = bcv
    return (_BOOK_ORDER.get(book, 10_000), book, ch, v)


# run once, near the top of cell 51
_VERSE_CACHE = {}
def _verse_index(df, name, text_col='form'):
    if name in _VERSE_CACHE:
        return _VERSE_CACHE[name]
    g = (df.groupby(['book', 'chapter', 'verse'])[text_col]
           .apply(lambda s: ' '.join(s.astype(str))))
    _VERSE_CACHE[name] = g.to_dict()
    return _VERSE_CACHE[name]

def _verse_text(df, book, ch, v, _name=None, text_col='form'):
    idx = _verse_index(df, _name or id(df), text_col)
    return idx.get((book, ch, v), '')


def _hb_text(book, ch, v):
    sub = hb_df[(hb_df['book'] == book) & (hb_df['chapter'] == ch) & (hb_df['verse'] == v)]
    if sub.empty:
        return ''
    return ' '.join(sub['form'].astype(str).tolist())

def _lxx_text(book, ch, v):
    sub = strongs_df[(strongs_df['book'] == book) & (strongs_df['chapter'] == ch) & (strongs_df['verse'] == v)]
    if not sub.empty:
        return ' '.join(sub['form'].astype(str).tolist())
    return ''

def _abp_text(book, ch, v):
    sub = abp_strongs_df[(abp_strongs_df['book'] == book) & (abp_strongs_df['chapter'] == ch) & (abp_strongs_df['verse'] == v)]
    if not sub.empty:
          return ' '.join(sub['form'].astype(str).tolist())
    return ''


def build_verse_dicts(bcv_set, needle,
                      needle_1694=None, needle_24=None,
                      needle_lxx = None, needle_abp = None,
                      needle_hb = None,
                      transform_1694= lambda x:x, transform_24=lambda x:x,
                      transform_65 = lambda x: x,
                      transform_lxx = lambda x: x,
                      transform_abp = lambda x: x,
                      transform_hb = lambda x: x,
                      highlight_accent_insensitive=True,
                      source='hb_mt'):
    """
    bcv_set: iterable of (book, chapter, verse) in the numbering system given by `source`.
    source:  'hb_mt' (keys match hb_df/strongs_df/abp_strongs_df/l1694_df) or
             'l65'  (keys match l65_df/l24_df). 2024 for now uses the same map as 1965.
    needle:  the 1965 / 2024 search term (e.g. 'tic' or 'mīl')
    needle_1694: variant for Glik (e.g. 'tiz' or 'mihl'); defaults to needle
    Returns a list of verse-dicts in canonical order, ready for chapter_to_html_render.
    """
    needle_1694 = needle_1694 if needle_1694 is not None else needle
    needle_24 = needle_24 if needle_24 is not None else needle

    # mapping-aware text fetchers; fall back to direct dict lookup (NT books,
    # verses not in any mapping) so behaviour for the untouched cases matches before.
    def _l65_for(book, ch, v):
        if source == 'hb_mt':
            try:
                return get_l65_text_for_hb_key((book, ch, v), l65_g)
            except (KeyError, Exception):
                pass
        return _verse_text(l65_df, book, ch, v)

    def _l24_for(book, ch, v):
        if source == 'hb_mt':
            try:
                # per user: l24 uses the same l65 map for now
                return get_l65_text_for_hb_key((book, ch, v), l24_g)
            except (KeyError, Exception):
                pass
        return _verse_text(l24_df, book, ch, v)

    def _1694_for(book, ch, v):
        if source == 'l65':
            try:
                return get_hb_keyed_text_for_l65_key((book, ch, v), l1694_g, sort_col='word_idx')
            except (KeyError, Exception):
                pass
        return _verse_text(l1694_df, book, ch, v)

    def _hb_for(book, ch, v):
        if source == 'l65':
            try:
                return get_hb_keyed_text_for_l65_key((book, ch, v), hb_g, sort_col='word')
            except (KeyError, Exception):
                pass
        return _hb_text(book, ch, v)

    def _lxx_for(book, ch, v):
        if source == 'l65':
            try:
                return get_hb_keyed_text_for_l65_key((book, ch, v), strongs_g, sort_col='word')
            except (KeyError, Exception):
                pass
        # even for source='hb_mt', LXX has its own splits vs MT — use helper when possible
        try:
            return get_greek_text_for_hb_key((book, ch, v), strongs_g, strongs_g)
        except (KeyError, Exception):
            return _lxx_text(book, ch, v)

    def _abp_for(book, ch, v):
        if source == 'l65':
            try:
                return get_hb_keyed_text_for_l65_key((book, ch, v), abp_strongs_g, sort_col='word')
            except (KeyError, Exception):
                pass
        try:
            return get_greek_text_for_hb_key((book, ch, v), abp_strongs_g, abp_strongs_g)
        except (KeyError, Exception):
            return _abp_text(book, ch, v)

    rows = []
    for book, ch, v in sorted(bcv_set, key=_sort_key):
        t65   = _l65_for(book, ch, v)
        t24   = _l24_for(book, ch, v)
        t1694 = _1694_for(book, ch, v)
        thb   = _hb_for(book, ch, v)
        tlxx  = _lxx_for(book, ch, v)
        tabp  = _abp_for(book, ch, v)

        rows.append({
            'book': book,
            'chapter': ch,
            'verse': v,
            'hebrew_text_full_original':        render_verse(thb, needle_hb, transform_hb),
            'lxx_text_full_original':           render_verse(tlxx, needle_lxx, transform_lxx),
            'abp_text_full_original':           render_verse(tabp, needle_abp, transform_abp),
            'latvian_text_full_original_65':    render_verse(t65,   needle,      transform_65),
            'latvian_text_full_original_24':    render_verse(t24,   needle_24,   transform_24),
            'latvian_text_full_original_1694':  render_verse(t1694, needle_1694, transform_1694),
        })
    return rows


def group_by_chapter(verse_dicts):
    """Chunk a flat list of verse-dicts into per-(book,chapter) lists, preserving order."""
    out = []
    cur_key = None
    cur_list = []
    for d in verse_dicts:
        k = (d['book'], d['chapter'])
        if k != cur_key:
            if cur_list:
                out.append(cur_list)
            cur_list = []
            cur_key = k
        cur_list.append(d)
    if cur_list:
        out.append(cur_list)
    return out

In [47]:
# replacement for the broken renderResultsToHTML in cell 58

def renderResultsToHTML(verse_dicts, heading=None):
    html = getHead()
    if heading:
        html += f"<h1 style='text-align:center'>{heading}</h1>"
    for chap in group_by_chapter(verse_dicts):
        html += chapter_to_html_render(chap)
    html += getTail()
    return html


def render_comparison_triplet(set_a, set_b, needle,
                              titleNeedle=None,
                              name_a='A', name_b='B',
                              needle_1694=None, needle_24=None,
                              needle_lxx = None, needle_abp = None,
                              needle_hb = None,
                              transform_1694= lambda x:x, transform_24=lambda x:x,
                              transform_65 = lambda x: x,
                              transform_lxx = lambda x: x,
                              transform_abp = lambda x: x,
                              transform_hb = lambda x: x,
                              highlight_accent_insensitive=True,
                              source='hb_mt',
                              fileNamePrefix=""):
    """
    Build the three HTML views described in 2. vizualizācija:
      (kopīgie, tikai A, tikai B)
    """
    common    = set_a & set_b
    only_in_a = set_a - set_b
    only_in_b = set_b - set_a

    build = lambda s: build_verse_dicts(
        s, needle,
        needle_1694=needle_1694, needle_24=needle_24,
        needle_lxx = needle_lxx, needle_abp = needle_abp,
        needle_hb= needle_hb,
        highlight_accent_insensitive=highlight_accent_insensitive,
        transform_1694=transform_1694, transform_24=transform_24,
        transform_65=transform_65, transform_lxx=transform_lxx,
        transform_hb= transform_hb,
        transform_abp = transform_abp
    ,
        source=source
    )
    needle_for_title = titleNeedle or needle or needle_1694 or needle_24 or needle_lxx or needle_abp
    return {
        f'{fileNamePrefix} tilts {name_a} UN  {name_b} ({needle_for_title}) abos':       renderResultsToHTML(build(common),    heading=f'{needle!r}: kopīgie ({name_a} ∩ {name_b}) — {len(common)}'),
        f'{fileNamePrefix} tikai {name_a} NE {name_b} ({needle_for_title})':               renderResultsToHTML(build(only_in_a), heading=f'{needle!r}: tikai {name_a} NE {name_b} — {len(only_in_a)}'),
        f'{fileNamePrefix} tikai {name_b} NE {name_a} ({needle_for_title})':               renderResultsToHTML(build(only_in_b), heading=f'{needle!r}: tikai {name_b} NE {name_a} — {len(only_in_b)}'),
    }

In [48]:
import pathlib
def write_rendered(view):
  out_dir = pathlib.Path('render_out'); out_dir.mkdir(exist_ok=True)
  for title, html in view.items():
      safe = title.replace(' ', '_').replace('/', '_').replace('∩', 'and').replace("'", '')
      (out_dir / f'{safe}.html').write_text(html, encoding='utf-8')
      print(f'wrote {safe}.html  ({len(html):,} chars)')

# 2 -> call render (me)

In [49]:
def strip_diacritics_he(s):
    """Remove all Hebrew diacritics, vowels, cantillation — keep base letters only."""
    return ''.join(c for c in s if '\u05D0' <= c <= '\u05EA')

In [50]:
from tqdm.notebook import tqdm
from tqdm import tqdm
set_γαπ_lxx_2 = verse_set(resultats_γαπ_lxx_2)
set_γαπ_abp_2 = verse_set(resultats_γαπ_abp_2)
for v1, v2, v3 in tqdm(zip([('LXX', 'ABP'),
                       #('1965', '1694'),
                      # ('2024', '1694')
                      ],
 [(set_γαπ_lxx_2, set_γαπ_abp_2),
  #(set_65_mil_4t, set_1694_mil_4t),
  #(set_24_mil_4t, set_1694_mil_4t),
  ],
  [(SEARCH2, SEARCH2),
   #(SEARCH3.replace('ļ', 'l'), SEARCH3.replace('ī', 'ih')),
   #(SEARCH3.replace('ļ', 'l'), SEARCH3.replace('ī', 'ih'))
   ])):

  vview = render_comparison_triplet(
      v2[0], v2[1],
      titleNeedle =v3[0],
      needle="mīl",
      needle_24="mīl",
      needle_1694="mihl",
      needle_hb = "אַהֲבָֽ",
       needle_lxx =v3[0],                              # 'mīl'
      needle_abp=v3[1],      # 'mihl'
      name_a=v1[0], name_b=v1[1],
      transform_65=lambda x: strip_latvian_deep(x).lower(),
      transform_24=lambda x: strip_latvian_deep(x).lower(),
      transform_1694=lambda x: x.lower().replace("ł", "l"),
      transform_lxx=lambda x: '' if x==None else strip_greek_diacritics(x).lower(),
      transform_abp=lambda x:  '' if x==None else strip_greek_diacritics(x).lower(),
      transform_hb = lambda x: strip_diacritics_he(x),
      highlight_accent_insensitive=False,          # 4s = case/accent sensitive
  )
  write_rendered(vview)



1it [00:16, 16.67s/it]

wrote _tilts_LXX_UN__ABP_(γαπ)_abos.html  (517,716 chars)
wrote _tikai_LXX_NE_ABP_(γαπ).html  (16,101 chars)
wrote _tikai_ABP_NE_LXX_(γαπ).html  (11,632 chars)


In [51]:
SEARCH4='δευσ'
resultats_γαπ_lxx_4 = gk_ic_lxx_tr.search(SEARCH4)
print(resultats_γαπ_lxx_4[:2])

resultats_γαπ_abp_4 = gk_ic_abp_tr.search(SEARCH4)
print(resultats_γαπ_abp_4[:2])

resultats_γαπ_un_4 = gk_ic_un_tr.search(SEARCH4)
print(resultats_γαπ_un_4[:2])

                 form strong_num strong_en_title     book  chapter  verse  \
6162        διώδευσεν       1353        V-AAI-3S  genesis       12      6   
6249  ἐστρατοπέδευσεν     4759.2        V-AAI-3S  genesis       12      9   

      word is_patched form_en strong_title  
6162     1        NaN     NaN          NaN  
6249     5       True     NaN          NaN  
       verse  word_idx                form             form_en strong_num  \
7678      19        13  κατεστρατοπέδευσαν        [4bivouacked     2689.2   
18900      6        31           διώδευσεν  [2traveled through       1353   

            strong_title      book  chapter  
7678          bivouacked    joshua        4  
18900  to travel through  jeremiah        2  
       verse  word          form                form_en  strong_num  \
27150      1     0  Διοδεύσαντες  Having passed through        1353   
81688     16     0     παιδεύσας       Having chastised        3811   

                                            stron

In [ ]:
set_γαπ_lxx_4 = verse_set(resultats_γαπ_lxx_4)
set_γαπ_abp_4 = verse_set(resultats_γαπ_abp_4)
for v1, v4, v3 in tqdm(zip([('LXX', 'ABP'),
                       #('1965', '1694'),
                      # ('4044', '1694')
                      ],
 [(set_γαπ_lxx_4, set_γαπ_abp_4),
  #(set_65_mil_4t, set_1694_mil_4t),
  #(set_44_mil_4t, set_1694_mil_4t),
  ],
  [(SEARCH4, SEARCH4),
   #(SEARCH3.replace('ļ', 'l'), SEARCH3.replace('ī', 'ih')),
   #(SEARCH3.replace('ļ', 'l'), SEARCH3.replace('ī', 'ih'))
   ])):

  vview = render_comparison_triplet(
      v4[0], v4[1],
      titleNeedle =v3[0],
      needle="mīl",
      needle_44="prec",
      needle_1694="prez",
      needle_hb = "בָעַ֖ל",
       needle_lxx =v3[0],                              # 'mīl'
      needle_abp=v3[1],      # 'mihl'
      name_a=v1[0], name_b=v1[1],
      transform_65=lambda x: strip_latvian_deep(x).lower(),
      transform_44=lambda x: strip_latvian_deep(x).lower(),
      transform_1694=lambda x: x.lower().replace("ł", "l"),
      transform_lxx=lambda x: '' if x==None else strip_greek_diacritics(x).lower(),
      transform_abp=lambda x:  '' if x==None else strip_greek_diacritics(x).lower(),
      transform_hb = lambda x: strip_diacritics_he(x),
      highlight_accent_insensitive=False,          # 4s = case/accent sensitive
  )
  write_rendered(vview)